### Exploring Staple Food Price Dataset

In [43]:
import pandas as pd

# Load the original raw food price dataset
df = pd.read_csv(
    "data/raw/ethiopian_agriculture/02_staple_food_price.csv"
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (77017, 16)
Columns: ['country', 'market', 'admin_1', 'longitude', 'latitude', 'cpcv2', 'product', 'source_document', 'period_date', 'price_type', 'product_source', 'unit', 'unit_type', 'currency', 'value', 'Unnamed: 15']


In [44]:
print(df.dtypes)

country                str
market                 str
admin_1                str
longitude          float64
latitude           float64
cpcv2                  str
product                str
source_document        str
period_date            str
price_type             str
product_source         str
unit                   str
unit_type              str
currency               str
value              float64
Unnamed: 15        float64
dtype: object


In [45]:
df["period_date"] = pd.to_datetime(df["period_date"])

print("Date range:")
print(df["period_date"].min(), "to", df["period_date"].max())

Date range:
2020-01-01 00:00:00 to 2023-06-21 00:00:00


In [46]:
df = df.drop(columns=["Unnamed: 15"])

print("Shape after removing empty column:", df.shape)

Shape after removing empty column: (77017, 15)


In [47]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [48]:
summary = (
    df.groupby(["market", "product"])["value"]
      .agg(
          total="size",
          available="count"
      )
)

summary["missing"] = (
    summary["total"] - summary["available"]
)

summary["missing_pct"] = (
    summary["missing"] / summary["total"] * 100
)

print(summary.head())

                                                                    total  \
market               product                                                
Addis Ababa, Merkato Beans (Haricot)                                  182   
                     Camels (Local Quality)                           182   
                     Casual Labor (unskilled, daily, without food)    182   
                     Diesel                                           182   
                     Firewood                                         182   

                                                                    available  \
market               product                                                    
Addis Ababa, Merkato Beans (Haricot)                                      182   
                     Camels (Local Quality)                                 0   
                     Casual Labor (unskilled, daily, without food)        182   
                     Diesel                            

In [49]:
complete_missing = summary[
    summary["available"] == 0
].index

print(
    "Completely missing market-product combinations:",
    len(complete_missing)
)

Completely missing market-product combinations: 45


In [50]:
df = (
    df.set_index(["market", "product"])
      .drop(index=complete_missing)
      .reset_index()
)

print("Shape:", df.shape)
print("Remaining missing prices:", df["value"].isna().sum())

Shape: (68978, 15)
Remaining missing prices: 12606


In [51]:
# Recalculate missingness after removing completely empty series

summary = (
    df.groupby(["market", "product"])["value"]
      .agg(
          total="size",
          available="count"
      )
)

summary["missing"] = summary["total"] - summary["available"]

summary["missing_pct"] = (
    summary["missing"] / summary["total"] * 100
)

print(summary["missing_pct"].describe())

count    395.000000
mean      18.977152
std       28.168080
min        0.000000
25%        0.549451
50%        2.197802
75%       29.120879
max       99.450549
Name: missing_pct, dtype: float64


In [52]:
print("≤25% missing:",
      (summary["missing_pct"] <= 25).sum())

print("25–50% missing:",
      ((summary["missing_pct"] > 25) &
       (summary["missing_pct"] <= 50)).sum())

print(">50% missing:",
      (summary["missing_pct"] > 50).sum())

≤25% missing: 292
25–50% missing: 30
>50% missing: 73


In [53]:
# Make sure observations are in time order
df = df.sort_values(
    ["market", "product", "period_date"]
).reset_index(drop=True)


def longest_missing_run(series):
    groups = series.ne(series.shift()).cumsum()
    return series.groupby(groups).sum().max()


gap_info = (
    df.groupby(["market", "product"])["value"]
      .agg(
          total_missing=lambda x: x.isna().sum(),
          longest_missing_gap=lambda x: longest_missing_run(x.isna())
      )
      .sort_values("longest_missing_gap", ascending=False)
)

print(gap_info.head(20))

                                         total_missing  longest_missing_gap
market            product                                                  
Sodo              Sorghum (Yellow)                 176                  173
Degehabour        Horse beans                      171                  171
                  Beans (Haricot)                  171                  171
Jinka             Sorghum (Yellow)                 181                  168
Beddenno          Wheat Flour                      150                  150
Dire Dawa, Kezira Sorghum (Yellow)                 153                  147
Shire             Rice (Milled)                    144                  144
Mekele            Beans (Haricot)                  172                  142
Dire Dawa, Kezira Firewood                         139                  139
Sikela            Sorghum (Yellow)                 155                  135
Mekele            Diesel                           121                  121
            

In [54]:
# Check the distribution of longest missing gaps

print(gap_info["longest_missing_gap"].describe())

print("\nNumber of series by longest missing gap:")

print(
    "0 days/observations:",
    (gap_info["longest_missing_gap"] == 0).sum()
)

print(
    "1–4 observations:",
    gap_info["longest_missing_gap"].between(1, 4).sum()
)

print(
    "5–20 observations:",
    gap_info["longest_missing_gap"].between(5, 20).sum()
)

print(
    "More than 20 observations:",
    (gap_info["longest_missing_gap"] > 20).sum()
)

count    395.000000
mean      27.716456
std       43.854520
min        0.000000
25%        1.000000
50%        2.000000
75%       34.000000
max      173.000000
Name: longest_missing_gap, dtype: float64

Number of series by longest missing gap:
0 days/observations: 84
1–4 observations: 148
5–20 observations: 33
More than 20 observations: 130


In [55]:
# Identify series with short and long missing gaps

short_gap_series = gap_info[
    gap_info["longest_missing_gap"] <= 20
].index

long_gap_series = gap_info[
    gap_info["longest_missing_gap"] > 20
].index

print("Series with short gaps (≤20):", len(short_gap_series))
print("Series with long gaps (>20):", len(long_gap_series))

Series with short gaps (≤20): 265
Series with long gaps (>20): 130


In [56]:
# Display the series with the longest missing gaps

print(
    gap_info[
        gap_info["longest_missing_gap"] > 20
    ].head(30)
)

                                         total_missing  longest_missing_gap
market            product                                                  
Sodo              Sorghum (Yellow)                 176                  173
Degehabour        Horse beans                      171                  171
                  Beans (Haricot)                  171                  171
Jinka             Sorghum (Yellow)                 181                  168
Beddenno          Wheat Flour                      150                  150
Dire Dawa, Kezira Sorghum (Yellow)                 153                  147
Shire             Rice (Milled)                    144                  144
Mekele            Beans (Haricot)                  172                  142
Dire Dawa, Kezira Firewood                         139                  139
Sikela            Sorghum (Yellow)                 155                  135
Mekele            Diesel                           121                  121
            

In [57]:
# Check missing values in series with short gaps only

short_gap_data = (
    df.set_index(["market", "product"])
      .loc[short_gap_series]
      .reset_index()
)

print("Rows:", len(short_gap_data))
print("Missing values:", short_gap_data["value"].isna().sum())

Rows: 47219
Missing values: 1077


In [58]:
# Check whether short gaps are internal, beginning, or ending gaps

def gap_position(group):
    missing = group["value"].isna()

    if not missing.any():
        return "no_missing"

    first_valid = group["value"].first_valid_index()
    last_valid = group["value"].last_valid_index()

    first_missing = missing.idxmax()
    last_missing = missing[::-1].idxmax()

    if first_missing < first_valid:
        return "leading"

    if last_missing > last_valid:
        return "trailing"

    return "internal"


gap_position_summary = (
    short_gap_data
    .groupby(["market", "product"])
    .apply(gap_position)
    .value_counts()
)

print(gap_position_summary)

internal      125
no_missing     84
leading        53
trailing        3
Name: count, dtype: int64


In [59]:
# Select only series with internal gaps

internal_series = (
    gap_position_summary
)

internal_series = gap_position_summary[
    gap_position_summary.index == "internal"
]

print("Internal-gap series:", internal_series)

Internal-gap series: internal    125
Name: count, dtype: int64


In [60]:
# Get market-product series that have internal missing gaps

internal_series = []

for (market, product), group in short_gap_data.groupby(
    ["market", "product"]
):
    missing = group["value"].isna()

    if not missing.any():
        continue

    first_valid = group["value"].first_valid_index()
    last_valid = group["value"].last_valid_index()

    first_missing = missing.idxmax()
    last_missing = missing[::-1].idxmax()

    if first_valid < first_missing and last_missing < last_valid:
        internal_series.append((market, product))

print("Internal-gap series:", len(internal_series))
print(internal_series[:10])

Internal-gap series: 125
[('Bahir Dar', 'Beans (Haricot)'), ('Bahir Dar', 'Casual Labor (unskilled, daily, without food)'), ('Bahir Dar', 'Diesel'), ('Bahir Dar', 'Firewood'), ('Bahir Dar', 'Gasoline'), ('Bahir Dar', 'Goats (Local Quality)'), ('Bahir Dar', 'Horse beans'), ('Bahir Dar', 'Maize Grain (White)'), ('Bahir Dar', 'Mixed Teff'), ('Bahir Dar', 'Oxen (Local Quality)')]


In [61]:
# Fill only internal missing price values using time interpolation

df["period_date"] = pd.to_datetime(df["period_date"])

df = df.sort_values(
    ["market", "product", "period_date"]
).reset_index(drop=True)

before_missing = df["value"].isna().sum()

for market_product in internal_series:
    market, product = market_product

    mask = (
        (df["market"] == market) &
        (df["product"] == product)
    )

    df.loc[mask, "value"] = (
        df.loc[mask]
          .set_index("period_date")["value"]
          .interpolate(method="time")
          .values
    )

after_missing = df["value"].isna().sum()

print("Missing before:", before_missing)
print("Missing after:", after_missing)
print("Values filled:", before_missing - after_missing)

Missing before: 12606
Missing after: 11873
Values filled: 733


In [62]:
# Find leading and trailing missing values

remaining_df = df.sort_values(
    ["market", "product", "period_date"]
).reset_index(drop=True)

leading_missing = 0
trailing_missing = 0

leading_series = []
trailing_series = []

for (market, product), group in remaining_df.groupby(["market", "product"]):

    values = group["value"].reset_index(drop=True)
    missing = values.isna()

    if not missing.any():
        continue

    first_valid = values.first_valid_index()
    last_valid = values.last_valid_index()

    # Leading missing values
    if first_valid > 0:
        count = first_valid
        leading_missing += count
        leading_series.append((market, product, count))

    # Trailing missing values
    if last_valid < len(values) - 1:
        count = len(values) - 1 - last_valid
        trailing_missing += count
        trailing_series.append((market, product, count))

print("Leading missing values:", leading_missing)
print("Trailing missing values:", trailing_missing)

print("\nLeading-gap series:", len(leading_series))
print("Trailing-gap series:", len(trailing_series))

Leading missing values: 2047
Trailing missing values: 1264

Leading-gap series: 86
Trailing-gap series: 21


In [63]:
# Show leading-gap series

leading_df = pd.DataFrame(
    leading_series,
    columns=["market", "product", "missing_count"]
).sort_values(
    "missing_count",
    ascending=False
)

print("Leading gaps:")
print(leading_df.head(30))

Leading gaps:
               market                 product  missing_count
19         Degehabour             Horse beans            171
18         Degehabour         Beans (Haricot)            171
13           Beddenno             Wheat Flour            150
31  Dire Dawa, Kezira        Sorghum (Yellow)            147
67              Shire           Rice (Milled)            144
27  Dire Dawa, Kezira                Firewood            139
45              Logia         Beans (Haricot)            119
48              Logia           Sorghum (Red)            114
47              Logia           Rice (Milled)            113
46              Logia                Firewood            113
3               Awash                Firewood            113
4               Awash         Sorghum (White)            113
85             Yabelo        Sorghum (Yellow)            107
35           Gambella         Sorghum (White)             80
49             Mekele         Beans (Haricot)             30
76        

In [64]:
# Show trailing-gap series

trailing_df = pd.DataFrame(
    trailing_series,
    columns=["market", "product", "missing_count"]
).sort_values(
    "missing_count",
    ascending=False
)

print("\nTrailing gaps:")
print(trailing_df.head(30))


Trailing gaps:
        market               product  missing_count
16        Sodo      Sorghum (Yellow)            173
9        Jinka      Sorghum (Yellow)            168
10      Mekele       Beans (Haricot)            142
13      Sikela      Sorghum (Yellow)            135
3   Degehabour  Oxen (Local Quality)            106
8     Gambella       Sorghum (White)            101
7     Gambella         Sorghum (Red)             84
20      Yabelo      Sorghum (Yellow)             74
4       Dessie       Beans (Haricot)             50
2     Beddenno       Sorghum (White)             43
5       Dessie         Sorghum (Red)             42
15        Sodo       Sorghum (White)             35
11      Sekota         Rice (Milled)             31
0    Bahir Dar       Sorghum (White)             27
17      Warder            Mixed Teff             24
18      Warder      Sorghum (Yellow)             22
6       Dessie       Sorghum (White)              3
1     Beddenno       Beans (Haricot)            

In [65]:
# Recalculate longest missing gap for every market-product series

gap_info = (
    df.groupby(["market", "product"])["value"]
      .agg(
          total_missing=lambda x: x.isna().sum(),
          longest_missing_gap=lambda x: longest_missing_run(x.isna())
      )
      .sort_values("longest_missing_gap", ascending=False)
)

# Series with a missing gap longer than 20 observations
remove_series = gap_info[
    gap_info["longest_missing_gap"] > 20
].index

print("Series to remove:", len(remove_series))
print("\nLargest gaps:")
print(gap_info.loc[remove_series].head(30))

Series to remove: 130

Largest gaps:
                                                                 total_missing  \
market            product                                                        
Sodo              Sorghum (Yellow)                                         176   
Degehabour        Beans (Haricot)                                          171   
                  Horse beans                                              171   
Jinka             Sorghum (Yellow)                                         181   
Beddenno          Wheat Flour                                              150   
Dire Dawa, Kezira Sorghum (Yellow)                                         153   
Shire             Rice (Milled)                                            144   
Mekele            Beans (Haricot)                                          172   
Dire Dawa, Kezira Firewood                                                 139   
Sikela            Sorghum (Yellow)                           

In [66]:
# Remove market-product series with very long missing gaps

df = (
    df.set_index(["market", "product"])
      .drop(index=remove_series)
      .reset_index()
)

print("Shape after removing long-gap series:", df.shape)
print("Remaining missing values:", df["value"].isna().sum())

Shape after removing long-gap series: (47219, 15)
Remaining missing values: 344


In [67]:
# Analyze remaining missing values

remaining_missing = (
    df[df["value"].isna()]
    .groupby(["market", "product"])
    .size()
    .sort_values(ascending=False)
)

print("Total remaining missing values:", remaining_missing.sum())
print("\nMissing values by series:")
print(remaining_missing)

Total remaining missing values: 344

Missing values by series:
market                product                                      
Jinka                 Beans (Haricot)                                  66
Dessie                Sorghum (White)                                  33
Sodo                  Sorghum (Red)                                    33
Beddenno              Beans (Haricot)                                  20
                      Refined sugar                                    14
                      Rice (Milled)                                    14
                      Refined Vegetable Oil                            14
Bahir Dar             Refined sugar                                    10
Dessie                Rice (Milled)                                     8
Jinka                 Refined sugar                                     8
Dessie                Refined Vegetable Oil                             7
                      Refined sugar                    

In [68]:
# Check the longest remaining missing gap

gap_info_remaining = (
    df.groupby(["market", "product"])["value"]
      .agg(
          total_missing=lambda x: x.isna().sum(),
          longest_missing_gap=lambda x: longest_missing_run(x.isna())
      )
      .sort_values("longest_missing_gap", ascending=False)
)

print(gap_info_remaining[
    gap_info_remaining["total_missing"] > 0
])

                                                                    total_missing  \
market               product                                                        
Beddenno             Rice (Milled)                                             14   
                     Beans (Haricot)                                           20   
                     Refined Vegetable Oil                                     14   
                     Refined sugar                                             14   
Jinka                Refined sugar                                              8   
Dessie               Sorghum (White)                                           33   
Jinka                Beans (Haricot)                                           66   
Bahir Dar            Refined sugar                                             10   
Dessie               Refined Vegetable Oil                                      7   
Sodo                 Sorghum (Red)                               

In [69]:
# Fill remaining small leading and trailing gaps

df = df.sort_values(
    ["market", "product", "period_date"]
).reset_index(drop=True)

before = df["value"].isna().sum()

# Fill leading gaps using the first available value
df["value"] = (
    df.groupby(["market", "product"])["value"]
      .transform(lambda x: x.bfill())
)

# Fill trailing gaps using the last available value
df["value"] = (
    df.groupby(["market", "product"])["value"]
      .transform(lambda x: x.ffill())
)

after = df["value"].isna().sum()

print("Missing before:", before)
print("Missing after:", after)
print("Values filled:", before - after)

Missing before: 344
Missing after: 0
Values filled: 344


In [70]:
# Final missing-value check

print("Remaining missing values:", df["value"].isna().sum())

Remaining missing values: 0


In [71]:
print(df["value"].isna().sum())

0


In [72]:
# Check price values

print("Minimum price:", df["value"].min())
print("Maximum price:", df["value"].max())

print("\nZero prices:", (df["value"] == 0).sum())
print("Negative prices:", (df["value"] < 0).sum())

Minimum price: 1.234
Maximum price: 82520.0

Zero prices: 0
Negative prices: 0


In [73]:
# Check data types

print(df.dtypes)

market                        str
product                       str
country                       str
admin_1                       str
longitude                 float64
latitude                  float64
cpcv2                         str
source_document               str
period_date        datetime64[us]
price_type                    str
product_source                str
unit                          str
unit_type                     str
currency                      str
value                     float64
dtype: object


In [74]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [77]:
# Show the highest prices

print(
    df.nlargest(20, "value")[
        ["market", "product", "period_date", "unit",
         "price_type", "product_source", "currency", "value"]
    ]
)

                     market                 product period_date unit  \
12307            Degehabour  Camels (Local Quality)  2020-01-15   ea   
5214              Bahir Dar    Oxen (Local Quality)  2023-06-21   ea   
5211              Bahir Dar    Oxen (Local Quality)  2023-05-31   ea   
28537       Nazareth, Adama    Oxen (Local Quality)  2023-03-15   ea   
5205              Bahir Dar    Oxen (Local Quality)  2023-04-19   ea   
5206              Bahir Dar    Oxen (Local Quality)  2023-04-26   ea   
5212              Bahir Dar    Oxen (Local Quality)  2023-06-07   ea   
5209              Bahir Dar    Oxen (Local Quality)  2023-05-17   ea   
5213              Bahir Dar    Oxen (Local Quality)  2023-06-14   ea   
5208              Bahir Dar    Oxen (Local Quality)  2023-05-10   ea   
5207              Bahir Dar    Oxen (Local Quality)  2023-05-03   ea   
5210              Bahir Dar    Oxen (Local Quality)  2023-05-24   ea   
28538       Nazareth, Adama    Oxen (Local Quality)  2023-03-22 

In [79]:
group_cols = ["market", "product", "unit"]

q1 = df.groupby(group_cols)["value"].transform("quantile", 0.25)
q3 = df.groupby(group_cols)["value"].transform("quantile", 0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df["is_outlier"] = (
    (df["value"] < lower) |
    (df["value"] > upper)
)

print("Total potential outliers:", df["is_outlier"].sum())

Total potential outliers: 752


In [80]:
outliers = df[df["is_outlier"]].copy()

print(outliers[
    ["market", "product", "period_date", "unit", "price_type", "value"]
].sort_values("value", ascending=False).head(30))

                market                 product period_date unit price_type  \
12307       Degehabour  Camels (Local Quality)  2020-01-15   ea     Retail   
28537  Nazareth, Adama    Oxen (Local Quality)  2023-03-15   ea     Retail   
21029             Gode  Camels (Local Quality)  2023-05-24   ea     Retail   
34572       Shashemene    Oxen (Local Quality)  2022-09-07   ea     Retail   
34575       Shashemene    Oxen (Local Quality)  2022-09-28   ea     Retail   
21030             Gode  Camels (Local Quality)  2023-05-31   ea     Retail   
21031             Gode  Camels (Local Quality)  2023-06-07   ea     Retail   
12485       Degehabour  Camels (Local Quality)  2023-06-14   ea     Retail   
21032             Gode  Camels (Local Quality)  2023-06-14   ea     Retail   
21033             Gode  Camels (Local Quality)  2023-06-21   ea     Retail   
12479       Degehabour  Camels (Local Quality)  2023-05-03   ea     Retail   
22303             Gode    Oxen (Local Quality)  2023-05-24   ea 